# AutoGluon multimodal — 코랩 GPU

**목적** — 리뷰 **원문**을 언어모델에 직접 읽히면, 미리 뽑아둔 임베딩보다 나은가?

| 비교 대상 (로컬에서 측정, 영어 67,112행) | 랜덤분할 AUC |
|---|---|
| 부스팅 (숫자만) | 0.813 |
| MLP (숫자만) | 0.807 |
| **MLP (숫자 + 고정 임베딩)** | **0.777** ← 이걸 이기는지 본다 |

**왜 코랩인가** — AutoGluon은 CUDA만 GPU로 인식합니다. 맥의 MPS는 안 봅니다.
맥에서 돌리면 CPU로 3~6시간, 코랩 T4에서는 20~40분입니다.

---

## 시작 전 반드시

**런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 로 설정하세요.
안 하면 CPU로 돌아가서 몇 시간 걸립니다.

## 1. GPU 확인 — 여기서 실패하면 아래를 돌려도 소용없습니다

In [ ]:
!nvidia-smi
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU가 없습니다. 런타임 유형을 T4 GPU로 바꾸세요.'

## 2. AutoGluon 설치 (5~10분)

설치가 끝나면 **런타임을 다시 시작하라는 메시지**가 뜰 수 있습니다. 뜨면 다시 시작하고
**1번부터 다시** 실행하세요 (3번 이후만 다시 돌리면 됩니다).

In [ ]:
!pip install -q autogluon.multimodal
print('설치 완료')

## 3. 구글 드라이브 연결

`dataset.csv` 를 드라이브 어디에 올렸든 **알아서 찾습니다.**
경로를 직접 적고 싶으면 `DATASET` 한 줄만 고치세요.

| 드라이브에 올린 위치 | 경로 |
|---|---|
| 내 드라이브 최상단 | `/content/drive/MyDrive/dataset.csv` |
| `skn35-2nd-project` 폴더 안 | `/content/drive/MyDrive/skn35-2nd-project/dataset.csv` |
| 공유 드라이브 | `/content/drive/Shareddrives/팀이름/dataset.csv` |

> 코랩은 리눅스라 **대소문자를 구분합니다.** `SKN35` 와 `skn35` 는 다른 폴더입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

# 본인이 올린 위치. 모르면 그대로 두세요 — 아래에서 알아서 찾습니다.
DATASET = '/content/drive/MyDrive/skn35-2nd-project/dataset.csv'

if not os.path.exists(DATASET):
    print(f'적어둔 경로에 없습니다: {DATASET}')
    print('드라이브 전체에서 dataset.csv 를 찾는 중...\n')
    found = glob.glob('/content/drive/MyDrive/**/dataset.csv', recursive=True)
    if found:
        for f in found:
            print(f'  발견: {f}  ({round(os.path.getsize(f)/1024**2)} MB)')
        DATASET = found[0]
        print(f'\n→ 이걸로 진행합니다: {DATASET}')
    else:
        print('  드라이브에 dataset.csv 가 없습니다.')
        print('  아래 목록에서 올린 위치를 확인하세요:')
        for f in sorted(os.listdir('/content/drive/MyDrive'))[:40]:
            print('   ', f)

assert os.path.exists(DATASET), 'dataset.csv 를 찾지 못했습니다. 드라이브에 올렸는지 확인하세요.'
print(f'\n파일 확인 ✅  {round(os.path.getsize(DATASET)/1024**2)} MB')

## 4. 데이터 준비 — 로컬과 **똑같은** 분할

여기가 가장 중요합니다. 분할이 조금이라도 다르면 로컬 숫자와 비교가 무의미해집니다.

- 영어만 남기고 **index를 0부터 다시 매김** (로컬 `load_english()` 와 동일)
- `random_state=42`, `test_size=0.2`, 층화 (로컬 `make_splits()` 와 동일)

같은 `dataset.csv` 를 쓰면 시드가 고정돼 있어 **자동으로 같은 분할**이 나옵니다.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
META = ['recommendationid','appid','steamid','game','churn','review']

df = pd.read_csv(DATASET, encoding='utf-8', low_memory=False)
d  = df[df.language == 'english'].reset_index(drop=True)   # ★ index 재설정
y  = d['churn'].to_numpy()
g  = d['game'].to_numpy()

print(f'영어 {len(d):,}행 | 이탈률 {y.mean():.4f} | 게임 {d.game.nunique()}개')
assert len(d) == 67112, f'행 수가 다릅니다 ({len(d)}). dataset.csv 버전을 확인하세요'
assert abs(y.mean() - 0.4024) < 1e-3, '이탈률이 다릅니다. dataset.csv 버전 확인'
print('로컬과 동일한 데이터 확인 ✅')

tr, te = train_test_split(np.arange(len(d)), test_size=0.2,
                          random_state=SEED, stratify=y)
print(f'학습 {len(tr):,} / 시험 {len(te):,}   (로컬: 53,689 / 13,423)')

### 모델에 넣을 재료

숫자 24개 + **리뷰 원문**. 임베딩을 미리 뽑지 않습니다 — AutoGluon이 글을 직접 읽습니다.

`game` 은 뺍니다(B셋 기준). 게임 이름을 주면 외워버립니다.

In [ ]:
FEATURES = [c for c in d.columns if c not in META]   # 숫자·범주 24개
X = d[FEATURES].copy()
X['review'] = d['review'].fillna('').astype(str)     # 리뷰 원문 추가
X['churn']  = y

print(f'재료 {X.shape[1]-1}열 = 숫자·범주 {len(FEATURES)} + 리뷰 원문 1')
print('빈 리뷰:', int((X.review.str.strip()=='').sum()), '건')
X.head(3)

## 5. 학습

`time_limit` 은 초 단위입니다. **1800(30분)** 부터 시작하세요.

| 시간 | 예상 |
|---|---|
| 1800 (30분) | T4에서 충분히 수렴 |
| 3600 (1시간) | 여유 있게 |

> 코랩 무료는 **90분 유휴 시 끊깁니다.** 탭을 열어두고 가끔 클릭하세요.

In [ ]:
import time
from autogluon.multimodal import MultiModalPredictor

TIME_LIMIT = 1800   # 초

t0 = time.time()
pred = MultiModalPredictor(label='churn', eval_metric='roc_auc',
                           problem_type='binary', path='/content/ag_mm')
pred.fit(X.iloc[tr], time_limit=TIME_LIMIT, presets='medium_quality')
print(f'\n학습 {time.time()-t0:.0f}초')

## 6. 채점 — 로컬과 같은 지표

**정확도는 쓰지 않습니다.** 이탈률이 40.2%라 전부 '잔존'만 찍어도 59.8%가 나옵니다.

In [ ]:
from sklearn.metrics import (average_precision_score, f1_score,
                             precision_recall_curve, recall_score, roc_auc_score)

proba = pred.predict_proba(X.iloc[te]).iloc[:, 1].to_numpy()
yt = y[te]
hard = (proba >= 0.5).astype(int)

prec, rec, thr = precision_recall_curve(yt, proba)
f1s = np.divide(2*prec*rec, prec+rec, out=np.zeros_like(prec), where=(prec+rec)>0)
best = int(np.argmax(f1s))

row = {
    '모델명': 'AutoGluon(multimodal)', '변수묶음': 'B셋+원문', '분할방식': '랜덤',
    'AUC': round(roc_auc_score(yt, proba), 4),
    'PR-AUC': round(average_precision_score(yt, proba), 4),
    'Recall': round(recall_score(yt, hard, zero_division=0), 4),
    'F1': round(f1_score(yt, hard, zero_division=0), 4),
    '편차': 0.0,
    'best_F1': round(float(f1s[best]), 4),
    'best_임계값': round(float(thr[best]) if best < len(thr) else 1.0, 3),
    '학습시간': f'{TIME_LIMIT}s', '전처리버전': 'v1.2', '행수': len(d),
    '시각': pd.Timestamp.now().strftime('%m-%d %H:%M'), '메모': '코랩 T4 · 글 원문 직접',
}
print(pd.Series(row).to_string())

### 로컬 결과와 비교

In [ ]:
비교 = pd.DataFrame([
    ['부스팅(숫자만)',        0.8129],
    ['MLP(숫자만)',          0.8068],
    ['MLP(숫자+고정임베딩)',  0.7770],
    ['AutoGluon(multimodal)', row['AUC']],
], columns=['모델', '랜덤분할 AUC'])
비교['고정임베딩 대비'] = (비교['랜덤분할 AUC'] - 0.7770).round(4)
print(비교.to_string(index=False))

diff = row['AUC'] - 0.7770
if diff > 0.02:
    print('\n→ 원문을 직접 읽는 게 확실히 낫다. 게임분할 5조각도 돌려볼 값어치가 있다')
elif diff > 0.005:
    print('\n→ 조금 낫다. 결과서에 적되 결론은 안 바뀐다')
else:
    print('\n→ 차이 없음. "글을 최선을 다해 읽어도 안 오른다"가 확인됐다.')
    print('   우리 결론(행동이 말보다 정직하다)이 더 단단해진다')

## 7. 결과 저장

**코랩의 `/content/` 아래는 세션이 끝나면 전부 사라집니다.**
남기고 싶은 것은 반드시 드라이브(`/content/drive/`)로 옮겨야 합니다.

| 무엇 | 크기 | 저장할까 |
|---|---|---|
| **결과 CSV** | 1KB | ✅ 항상 (로컬 results.csv 와 같은 형식) |
| 학습된 모델 | 수백 MB | 선택 — 비교 실험이라 보통 필요 없습니다 |

결과 CSV 는 `dataset.csv` 를 올려둔 폴더에 저장됩니다.

In [ ]:
import os

# dataset.csv 가 있던 폴더에 나란히 저장한다
OUT_DIR = os.path.dirname(DATASET)
OUT = os.path.join(OUT_DIR, 'autogluon_multimodal_result.csv')

pd.DataFrame([row]).to_csv(OUT, index=False, encoding='utf-8-sig')
print('결과 저장 ✅')
print(' ', OUT)
print(f'  ({round(os.path.getsize(OUT)/1024, 1)} KB — 드라이브라서 세션이 끝나도 남습니다)')

# ── 모델도 남기고 싶다면 (선택) ─────────────────────────────
# 비교 실험이라 보통 필요 없습니다. 수백 MB 라 업로드도 오래 걸립니다.
SAVE_MODEL = False

if SAVE_MODEL:
    import shutil
    dst = os.path.join(OUT_DIR, 'ag_multimodal_model')
    shutil.copytree('/content/ag_mm', dst, dirs_exist_ok=True)
    size = sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(dst) for f in fs) / 1024**2
    print(f'\n모델 저장 ✅  {dst}  ({size:.0f} MB)')
else:
    print('\n모델은 저장 안 함 (SAVE_MODEL = True 로 바꾸면 저장)')
    print('  → 세션이 끝나면 /content/ag_mm 은 사라집니다')

---

## 결과를 로컬로 가져오기

드라이브에서 `autogluon_multimodal_result.csv` 를 내려받은 뒤,
레포에서 이렇게 합칩니다.

```bash
# 받은 파일을 레포 results/ 에 두고
uv run python -c "
import pandas as pd
from src.config import RESULTS_CSV, ENC_READ, ENC_CSV
old = pd.read_csv(RESULTS_CSV, encoding=ENC_READ)
new = pd.read_csv('results/autogluon_multimodal_result.csv', encoding=ENC_READ)
pd.concat([old, new]).to_csv(RESULTS_CSV, index=False, encoding=ENC_CSV)
print('합침:', len(old), '->', len(old)+len(new), '줄')
"
```

그러면 `uv run python -c "from src.evaluate import summary; summary()"` 에
코랩 결과까지 한 표로 나옵니다.

## 세션이 끊겼다면

1번(GPU 확인)부터 다시 실행하세요. 드라이브의 `dataset.csv` 는 그대로 있으니
**다시 업로드할 필요는 없습니다.** 설치(2번)만 5~10분 다시 걸립니다.

## 무료로 부족하면

- GPU 할당을 계속 거절당하거나
- 게임분할 5조각까지 돌리고 싶을 때

이럴 때만 Colab Pro($9.99)를 고려하세요. **Pro도 브라우저를 닫으면 끊깁니다**
(진짜 백그라운드 실행은 Pro+ $49). 자면서 돌리는 게 목적이면 Pro는 도움이 안 됩니다.

---

# 8. (선택) 게임 단위 분할 — 밤새 돌리기

**처음 보는 게임에서도 원문 읽기가 유리한가?**

| 비교 대상 (로컬 측정) | 게임 단위 AUC |
|---|---|
| MLP (숫자 + 고정 임베딩) | 0.730 ± 0.028 |
| 우리 부스팅 (숫자 + 글) | 0.757 ± 0.041 |
| AutoGluon (tabular) | 0.762 ± 0.041 |
| **AutoGluon (multimodal)** | **← 이걸 잰다** |

## ⚠️ 자기 전에 반드시

| | |
|---|---|
| **맥이 잠들면 세션이 끊깁니다** | 터미널에 `caffeinate -d -i -s` 실행해두세요 |
| **덮개를 닫지 마세요** | 닫으면 잠자기 = 세션 종료 |
| **전원 연결** | 배터리로는 잠자기로 빠집니다 |
| **브라우저 탭 열어두기** | 닫으면 종료 (Pro+ 아니면) |

**조각마다 드라이브에 저장합니다.** 3조각에서 끊겨도 1·2조각 결과는 남습니다.

In [ ]:
N_FOLDS    = 3      # 3이면 1~1.5시간, 5면 2~3시간
SAMPLE     = 0      # 0 = 전체 67,112행. 20000 으로 줄이면 훨씬 빠름
TIME_LIMIT = 1800   # 조각당 제한(초)

print(f'조각 {N_FOLDS}개 · {"전체" if not SAMPLE else f"{SAMPLE:,}행"} · 조각당 {TIME_LIMIT}초')
print(f'예상 총 시간: 약 {N_FOLDS * TIME_LIMIT / 3600:.1f}시간 (학습만, 준비시간 별도)')

In [ ]:
import time, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (average_precision_score, f1_score,
                             precision_recall_curve, recall_score, roc_auc_score)
from autogluon.multimodal import MultiModalPredictor

Xg, yg, gg = X, y, g
if SAMPLE and SAMPLE < len(X):
    rs = np.random.RandomState(SEED)
    keep = np.sort(rs.choice(len(X), SAMPLE, replace=False))
    Xg, yg, gg = X.iloc[keep].reset_index(drop=True), y[keep], g[keep]
    print(f'{len(Xg):,}행으로 축소')

FOLD_OUT = os.path.join(OUT_DIR, 'multimodal_folds.csv')
rows = []

for k, (tr_i, te_i) in enumerate(GroupKFold(n_splits=N_FOLDS).split(Xg, yg, groups=gg), 1):
    print(f'\n{"="*60}\n조각 {k}/{N_FOLDS}  시험 게임 {len(set(gg[te_i]))}개 · {len(te_i):,}행')
    t0 = time.time()
    try:
        m = MultiModalPredictor(label='churn', eval_metric='roc_auc',
                                problem_type='binary', path=f'/content/ag_fold{k}')
        m.fit(Xg.iloc[tr_i], time_limit=TIME_LIMIT, presets='medium_quality')
        pr = m.predict_proba(Xg.iloc[te_i]).iloc[:, 1].to_numpy()
        yt = yg[te_i]
        rows.append({'조각': k, 'AUC': roc_auc_score(yt, pr),
                     'PR-AUC': average_precision_score(yt, pr),
                     'Recall': recall_score(yt, (pr>=.5).astype(int), zero_division=0),
                     'F1': f1_score(yt, (pr>=.5).astype(int), zero_division=0),
                     '시험게임수': len(set(gg[te_i])), '시험행수': len(te_i),
                     '초': round(time.time()-t0)})
        print(f'  ✅ AUC {rows[-1]["AUC"]:.4f}  ({rows[-1]["초"]}초)')
    except Exception as e:
        print(f'  ❌ 실패: {type(e).__name__}: {str(e)[:200]}')
        continue
    finally:
        # ★ 조각마다 저장한다. 뒤에서 끊겨도 여기까지는 남는다
        if rows:
            pd.DataFrame(rows).to_csv(FOLD_OUT, index=False, encoding='utf-8-sig')
            print(f'  💾 저장 ({len(rows)}조각) → {FOLD_OUT}')

print(f'\n{"="*60}\n완료: {len(rows)}/{N_FOLDS} 조각')

## 결과 정리 — 아침에 확인하세요

중간에 끊겼어도 이 셀만 다시 돌리면 저장된 조각까지 집계됩니다.

In [ ]:
f = pd.read_csv(FOLD_OUT, encoding='utf-8')
print(f.to_string(index=False))

mean, std = f.AUC.mean(), f.AUC.std(ddof=0)
print(f'\n게임 단위 AUC  {mean:.4f} ± {std:.4f}   ({len(f)}조각)')

print('\n로컬 결과와 비교')
cmp = pd.DataFrame([
    ['MLP(숫자+고정임베딩)',   0.7295],
    ['우리 부스팅(숫자+글)',    0.7567],
    ['AutoGluon(tabular)',   0.7620],
    ['AutoGluon(multimodal)', round(mean, 4)],
], columns=['모델', '게임분할 AUC'])
print(cmp.to_string(index=False))

row_g = {'모델명':'AutoGluon(multimodal)', '변수묶음':'B셋+원문',
         '분할방식':f'게임({len(f)})', 'AUC':round(mean,4),
         'PR-AUC':round(f['PR-AUC'].mean(),4), 'Recall':round(f.Recall.mean(),4),
         'F1':round(f.F1.mean(),4), '편차':round(std,4), 'best_F1':'', 'best_임계값':'',
         '학습시간':f'{int(f.초.sum())}s', '전처리버전':'v1.2', '행수':int(f.시험행수.sum()),
         '시각':pd.Timestamp.now().strftime('%m-%d %H:%M'),
         '메모':f'코랩 T4 · 게임분할 {len(f)}조각'}
OUT_G = os.path.join(OUT_DIR, 'autogluon_multimodal_group.csv')
pd.DataFrame([row_g]).to_csv(OUT_G, index=False, encoding='utf-8-sig')
print(f'\n저장 ✅ {OUT_G}')